# 05 — Drift Detection: Version Lineage as a Time Axis

**Tier 2 — Boundaries & robustness** · [GenAI Alignment scenario library](../README.md#scenario-library) · native — no sibling repo tests this

> **In one sentence:** does the same simulated system stay behaviorally stable as the model underneath it changes, or as its own instructions get edited, and would this harness actually notice if it didn't?

| | |
|---|---|
| **Risk if untested** | Outputs change over time with no input change, driven by silent model, tool, or prompt updates. |
| **What this tests** | Behavior is stable absent input change; any material change is detected, explained, and gated. |

**Why this scenario matters more than its Tier 2 placement suggests.** Every other scenario in this repo answers "is the system aligned *right now*." This one answers a different, arguably more consequential question for a governance program: **does a system that passed every other test yesterday still pass today, and would you know if it didn't?** A model-risk program that only ever tests at deployment time has no way to catch a vendor's silent model update, a prompt edit that shipped without review, or a floating alias quietly drifting — all real, ongoing risks, not hypotheticals (see the retirement finding below). This is the one scenario in the library explicitly designed to answer "would we know."

**The core design problem: a notebook run happens at one point in time, but "drift" is a question about behavior *over* time.** Waiting for real calendar time to pass isn't available in a single sitting, so this scenario resolves it two different ways for two different axes:

- **Model version as a time axis** — vendor-shipped dated snapshots of the same model family are themselves real, time-separated data points, available today without waiting. `DRIFT_MODEL_SEQUENCE` (`.env`) holds that lineage.
- **A live floating alias as a stand-in for silent/calendar drift** — `DRIFT_FLOATING_MODEL` adds a comparison against a live, undated, auto-updating deployment. It can't tell you *whether* it drifted over the last month (that needs an actual re-run a month from now), but it can tell you whether it *already* differs from what's currently pinned, today, with zero waiting.
- **Prompt wording as its own axis, independent of the model** — held-fixed-model comparisons that isolate a system-prompt edit from a version bump, since a prompt edit is arguably a *more common* real-world drift trigger than a model swap, and one the version axis alone would never catch.

**Five things measured here, all against the same HR/IT golden set [Intended Performance](../docs/intended_performance.md) already scores for correctness, plus a generic supplement added for statistical power — no new dataset authored for either:**

1. **Noise floor per version** — N repeats at every snapshot, so a real version-to-version difference and ordinary run-to-run stochastic noise aren't confused for each other.
2. **Cross-version drift scoring** — does a later version's output still match the baseline's own dominant answer, on both a score axis and a semantic axis, "material" only after correcting for testing multiple tasks at once (see Methodology).
3. **Public-benchmark supplement** — a second, generic dataset (MMLU/TriviaQA/ARC, already used elsewhere in this repo) run through the *same* version-sweep and floating-check tracks, purely to add statistical power to "did the model change" — not use-case-grounded, and not run through the prompt-drift or harness-validation tracks.
4. **Harness validation** — a drift detector that's never been shown to detect anything, or to stay quiet on nothing, isn't validated. Two synthetic controls against the same baseline snapshot confirm the pipeline actually has the power to do both.
5. **Prompt drift** — the model held fixed, only the system-prompt wording varying between the current prompt and a realistic (non-adversarial) candidate edit.

**A real finding from building this notebook, not a hypothetical:** 4 of 6 originally candidate dated snapshots had already been retired by this deployment before this scenario could test against them — confirmed via live calls returning HTTP 410, not assumed. See Methodology below.

This notebook is code-light — everything above lives in [`scenarios/drift_detection.py`](../scenarios/drift_detection.py).

## ⚙️ Setup

```bash
pip install -e .
cp .env.example .env   # then fill in DRIFT_MODEL_SEQUENCE (and optionally DRIFT_FLOATING_MODEL)
```

`DRIFT_MODEL_SEQUENCE` needs at least 2 dated snapshots from the same model family — see `.env.example` for the exact format. New here? See [README — Setup](../README.md#setup) first. The cell below checks what's actually present in *this* kernel and stops cleanly if anything's missing, rather than failing deep in a later cell after API calls have already started.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Run from the repo root so relative paths (fixtures, outputs) resolve the
# same way whether this notebook or a script calls the scenario module.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
load_dotenv(Path.cwd() / ".env")

from scenarios import drift_detection as scenario
from scenarios import intended_performance as ip_scenario
from reporting.html_report import embed_report, render_report, save_report
from reporting.env_check import check_environment
from reporting.artifacts import artifact_trail
from reporting.display import GENERIC_JUDGE_MODEL_NAME, GENERIC_PROVIDER_NAME

pd.set_option("display.max_colwidth", 160)

### Environment Check

In [ ]:
ready = check_environment(
    required_packages=["genai_capability_bench", "jinja2", "matplotlib"],
    required_env_vars=["OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_API_VERSION", "DRIFT_MODEL_SEQUENCE", "JUDGE_MODEL"],
)
assert ready, "Fix the items above before continuing — later cells will spend real API calls."

<a id="methodology"></a>
## 📐 Methodology

**The target system, shared with Intended Performance.** The exact same `RAG_SYSTEM_PROMPT` mandate plus per-question knowledge-base document, run against every version in the lineage — only the model deployment changes, one axis at a time.

**Why 2 live snapshots instead of the 6 originally planned.** The candidate lineage started as 6 dated GPT-5.x snapshots spanning ~8 months. A pre-build smoke test against every candidate found that 4 of them — the 4 oldest — now return HTTP 410 ("deployment retired") on this Azure deployment. That's not a workaround-able bug; it's the deployment itself no longer serving those snapshots. Rather than discard that as an inconvenience, it's kept as a documented finding: **a version lineage this scenario has to plan around a vendor retiring older pinned snapshots is itself evidence for the risk this scenario tests.** See [Limitations & Future Work](../docs/drift_detection.md#limitations--future-work).

**"Material drift" — the statistical bar every track uses, and why it changed twice.** A shift only counts as material when a **two-sample test** rejects "these two groups are the same" — a Welch-style z-test on the continuous score, a two-proportion z-test on the semantic match rate — using *both* groups' own sampling variance, not a raw diff and not baseline treated as a fixed, certain reference point. That itself replaced an earlier, simpler check (a confidence interval built from the candidate's own repeats, compared against baseline's bare point estimate) that the harness-validation controls below caught structurally false-flagging whenever baseline happened to be perfectly self-consistent — real data exposed a real design flaw, which is exactly what those controls exist to do. On top of the two-sample test, every drift table applies **Benjamini-Hochberg correction across all tasks tested at once** (same `benjamini_hochberg` function Consistency & Reliability already uses for its own per-task significance test) — testing 10 tasks simultaneously at an uncorrected 95% confidence level produces roughly 0.5 false positives *by chance alone*; without this correction, that expected noise gets misread as a finding.

**Four tracks against the HR/IT golden set, plus a fifth generic supplement:**

1. **Version sweep** — `scenario.N_REPEATS` repeats of the golden set against each live dated snapshot in `DRIFT_MODEL_SEQUENCE`, oldest first. `scenario.version_noise_floor` computes each version's own internal consistency (reusing `reporting/repeat_run.py`'s existing variance + bidirectional-entailment machinery, unchanged from Consistency & Reliability's use of the same functions).
2. **Cross-version drift scoring** — `scenario.score_drift_vs_baseline` compares each candidate version's answers against the earliest version's dominant answer, on a deterministic-score axis and a semantic-entailment axis, BH-corrected as described above.
3. **Floating-alias check** (optional, if `DRIFT_FLOATING_MODEL` is set) — the same drift-scoring pipeline again, comparing a live, undated, auto-updating alias against the most recent *pinned* snapshot — the closest this scenario gets to silent/calendar drift without literally waiting for real time to pass.
4. **Harness validation** — `scenario.run_control_validation` runs two synthetic batches against the *same* baseline snapshot: an unperturbed second batch (expect: no material drift — the false-positive check) and a batch run against a system prompt deliberately corrupted to double every cited policy number (expect: material drift on most tasks — the detection-power check). **"Harness" here just means this testing pipeline itself** — the run-N-repeats-then-score-then-decide-if-material apparatus, not a new agent framework — and the point of this track is answering a narrower question than "did the model drift": *does this test actually work?* Both controls go through the real drift-scoring pipeline above, not a separate stub.
5. **Prompt drift** — `scenario.run_prompt_drift_check` holds the model fixed (the most recent live pinned snapshot) and varies only the system-prompt wording, between the current `RAG_SYSTEM_PROMPT` and `scenario.PROMPT_DRIFT_V2_SYSTEM_PROMPT` — a *realistic* edit (source citation, explicit uncertainty, a reordered clause), not the deliberate corruption track 4 uses. The literal prompt text for both versions is printed in that section below, not just described.

**Public-benchmark supplement, for the version-sweep and floating-check tracks only.** 10 HR/IT tasks gives thin statistical power for the narrower question "did the model itself change" — a generic MMLU/TriviaQA/ARC sample (`scenarios/fixtures/public_benchmark_sample.jsonl`, already reused by Consistency & Reliability, no new authoring) adds 30 more tasks through the identical pipeline. This mirrors Consistency & Reliability's own precedent of keeping a generic track *alongside* a use-case-grounded one rather than replacing either — they answer genuinely different questions. It is deliberately **not** run through prompt drift or harness validation, both of which are inherently about *this* system's own prompt, not raw model behavior.

## 🗂️ Data

Reused, not new — the exact same 10-question HR/IT policy golden set [Intended Performance](../docs/intended_performance.md) scores for correctness, loaded directly from `scenarios.intended_performance`. The public-benchmark supplement (see Methodology) is loaded separately, later, only for the tracks that use it.

In [ ]:
golden_set = ip_scenario.load_golden_set()
print(f"HR/IT policy golden set — {len(golden_set)} tasks (reused from Intended Performance)")
golden_set[["task_id", "subcategory", "trap_type", "expected_output"]]

## 🕰️ Version Lineage

The actual sequence this run tests, pulled live from `.env` — dates are real (not confidential); only the underlying deployment name is masked, see [`reporting/display.py`](../reporting/display.py).

In [ ]:
sequence = scenario.load_version_sequence()
floating_entry = scenario.load_floating_entry(sequence)

rows = [{"label": e["label"], "date": e["date"].isoformat()} for e in sequence]
if floating_entry:
    rows.append({"label": floating_entry["label"], "date": "live / undated"})
pd.DataFrame(rows)

**What this table actually means, in plain terms — three things are being compared here, not a long timeline:**

| Label | What it is | Compared against |
|---|---|---|
| `v1` | The **oldest** entry in `DRIFT_MODEL_SEQUENCE` — a real, vendor-dated snapshot | — (this *is* the baseline) |
| `v2` | The **newest** entry in `DRIFT_MODEL_SEQUENCE` — a real, vendor-dated snapshot, dated `+Xd` after `v1` | `v1` — "did behavior change between these two dated snapshots?" |
| `floating` | The value in `DRIFT_FLOATING_MODEL` — a *live, undated, auto-updating* alias, not a fixed snapshot | `v2` (the newest pinned one, not `v1`) — "has today's live alias already diverged from what we're currently pinned to?" |

**Why only 2 pinned snapshots, not 3 or more.** The version lineage originally planned was 6 dated snapshots spanning ~8 months; a pre-build connectivity check found the 4 oldest already retired (HTTP 410) on this deployment — see Methodology above. Only 2 dated snapshots from this model family are actually callable right now. A 3rd pinned point isn't just unbuilt — **no 3rd dated snapshot is currently available** for this family to test against; the only other option at this version is the floating (undated) alias, which is why it plays a different role (compared against the most recent pinned point, not folded into the pinned lineage itself) rather than being treated as a 3rd equivalent pinned version.

**Why `floating` is compared against `v2`, not `v1`.** The question `floating` exists to answer is "has today's live deployment already drifted from what we're currently relying on" — the most recent pinned snapshot (`v2`) is what "currently relying on" means in practice, not the oldest one in the lineage.

## ▶️ Run — Version Sweep (HR/IT track)

**What this cell actually spends:** `N_REPEATS × len(golden_set) × len(sequence)` target API calls, plus a variable number of judge calls for the internal-consistency check on top.

In [ ]:
judge_client = scenario.build_judge_client(sequence[0]["deployment"])
display(Markdown(f"**LLM Provider:** {GENERIC_PROVIDER_NAME}  \n**Judge model:** `{GENERIC_JUDGE_MODEL_NAME}` — a separate, fixed deployment, independent of the version lineage above (see Methodology)"))

sweep_results = scenario.run_version_sweep(golden_set, sequence, n=scenario.N_REPEATS)
noise_floor = scenario.version_noise_floor(sweep_results, judge_client)
print(f"{len(sweep_results)} rows collected ({scenario.N_REPEATS} repeats \u00d7 {len(golden_set)} tasks \u00d7 {len(sequence)} version(s))")
noise_floor[["version_label", "task_id", "avg_score", "score_std", "pass_rate", "semantic_consistency"]]

## 📊 Cross-Version Drift Scoring (HR/IT track)

Every candidate version scored against the earliest (baseline) version. `material_drift` is `True` only after Benjamini-Hochberg correction across all 10 tasks in this table — see Methodology for why an uncorrected per-task test would overstate how many findings are real.

In [ ]:
sweep_drift = scenario.build_sweep_drift_table(sweep_results, noise_floor, judge_client)
sweep_drift

## 🔄 Floating-Alias Check (HR/IT track)

Skipped automatically if `DRIFT_FLOATING_MODEL` isn't set in `.env`. Computes the floating alias's own noise floor too (same function as the pinned versions above), so it can be plotted alongside them next.

In [ ]:
if floating_entry:
    floating_results = scenario.run_floating_check(golden_set, floating_entry, n=scenario.N_REPEATS)
    floating_noise_floor = scenario.version_noise_floor(floating_results, judge_client)
    reference_label = sequence[-1]["label"]
    reference_results = sweep_results[sweep_results["version_label"] == reference_label]
    reference_var = noise_floor[noise_floor["version_label"] == reference_label]
    floating_drift = scenario.build_reference_drift_table(
        floating_results, reference_results, reference_var, judge_client, floating_entry["label"],
    )
else:
    floating_results, floating_noise_floor, floating_drift = None, None, None
    print("DRIFT_FLOATING_MODEL not set \u2014 skipping the floating-alias check.")
floating_drift

## 📈 Version-Lineage Trajectory (HR/IT track)

The pinned lineage (`v1` → `v2`, real dates, solid line) and the floating alias (dashed, marked undated) on one chart — this is the visual answer to "how has behavior actually moved." The floating point is plotted *after* the pinned lineage but explicitly flagged as undated, so it's never mistaken for having a real calendar position on this timeline.

In [ ]:
trajectory_chart = scenario.plot_trajectory(noise_floor, floating_noise_floor)

## 🌐 Public-Benchmark Supplement — Version Sweep & Floating Check Only

A second, generic dataset (MMLU/TriviaQA/ARC, already reused by [Consistency & Reliability](../docs/consistency_reliability.md), no new authoring) run through the *same* version-sweep and floating-check pipeline as the HR/IT track above — purely to add statistical power to "did the model itself change," since 10 HR/IT tasks alone gives that narrower question thin power. This dataset has no system-prompt support (see [`scenarios/intended_performance.py`](../scenarios/intended_performance.py)'s own module docstring for why that track was retired there for correctness testing), so it can't test whether the system stays in scope of its HR/IT mandate — only whether raw model behavior moved. **Deliberately not run through prompt drift or harness validation**, both of which are inherently about *this* system's own prompt, not a generic capability probe.

In [ ]:
public_sample = scenario.load_public_benchmark_sample()
print(f"Public-benchmark supplement \u2014 {len(public_sample)} tasks (reused from Consistency & Reliability)")

public_sweep_results = scenario.run_public_benchmark_sweep(public_sample, sequence, n=scenario.N_REPEATS)
public_noise_floor = scenario.version_noise_floor(public_sweep_results, judge_client)
public_sweep_drift = scenario.build_sweep_drift_table(public_sweep_results, public_noise_floor, judge_client)

if floating_entry:
    public_floating_results = scenario.run_public_benchmark_floating_check(public_sample, floating_entry, n=scenario.N_REPEATS)
    public_floating_noise_floor = scenario.version_noise_floor(public_floating_results, judge_client)
    public_reference_results = public_sweep_results[public_sweep_results["version_label"] == reference_label]
    public_reference_var = public_noise_floor[public_noise_floor["version_label"] == reference_label]
    public_floating_drift = scenario.build_reference_drift_table(
        public_floating_results, public_reference_results, public_reference_var, judge_client, floating_entry["label"],
    )
else:
    public_floating_results, public_floating_noise_floor, public_floating_drift = None, None, None

public_trajectory_chart = scenario.plot_trajectory(public_noise_floor, public_floating_noise_floor)
public_sweep_drift

## 🧪 Harness Validation — Test the Control, Not the Calendar

Two synthetic batches against the *same* baseline snapshot, run through the identical drift-scoring pipeline used above: an unperturbed second batch (expect quiet) and a batch with a system prompt deliberately corrupted to double every cited policy number (expect most tasks flagged). The corruption text below is the literal instruction appended to the knowledge-base document for every task in the corrupted batch — not a paraphrase.

In [ ]:
print(scenario.INJECTED_DRIFT_SYSTEM_SUFFIX)

If either control doesn't land where expected, that's a caveat on trusting the version-sweep's own `material_drift` flags this run, not just a footnote — see Key Findings in the report below.

In [ ]:
control_results = scenario.run_control_validation(golden_set, sequence[0]["deployment"], n=scenario.N_REPEATS)

baseline_label = sequence[0]["label"]
baseline_results = sweep_results[sweep_results["version_label"] == baseline_label]
baseline_var = noise_floor[noise_floor["version_label"] == baseline_label]

unchanged_candidates = control_results[control_results["version_label"] == "control: unchanged (2nd batch)"]
corrupted_candidates = control_results[control_results["version_label"] == "control: injected corruption"]

unchanged_drift = scenario.build_reference_drift_table(
    unchanged_candidates, baseline_results, baseline_var, judge_client, "control: unchanged (2nd batch)",
)
corrupted_drift = scenario.build_reference_drift_table(
    corrupted_candidates, baseline_results, baseline_var, judge_client, "control: injected corruption",
)
control_chart = scenario.plot_control_validation(unchanged_drift, corrupted_drift)

## 📝 Prompt Drift

Same model throughout (the most recent live pinned snapshot) — only the system-prompt wording differs. Both versions' literal text, so there's no ambiguity about what changed:

In [ ]:
print("=== prompt v1 (current) ===")
print(ip_scenario.RAG_SYSTEM_PROMPT)
print()
print("=== prompt v2 (candidate edit) ===")
print(scenario.PROMPT_DRIFT_V2_SYSTEM_PROMPT)

The edit above does three things a real prompt-engineering pass might: moves the redirect instruction earlier, adds a citation requirement, adds an explicit "say so if the policy doesn't answer this" instruction. None of that is adversarial — the point is whether an *ordinary, well-intentioned* rewrite still measurably shifts behavior, not whether an obvious one does (that's what Harness Validation's corruption control is for).

In [ ]:
prompt_results = scenario.run_prompt_drift_check(golden_set, sequence[-1]["deployment"], n=scenario.N_REPEATS)

v1_results = prompt_results[prompt_results["version_label"] == "prompt v1 (current)"]
v1_noise_floor = scenario.version_noise_floor(v1_results, judge_client)
v2_results = prompt_results[prompt_results["version_label"] == "prompt v2 (candidate edit)"]

prompt_drift = scenario.build_reference_drift_table(
    v2_results, v1_results, v1_noise_floor, judge_client, "prompt v2 (candidate edit)",
)
prompt_chart = scenario.plot_drift_by_task(
    prompt_drift,
    "Prompt-edit drift: current wording vs. a realistic v2 rewrite",
    "Same model throughout \u2014 only the system-prompt wording differs. A flag here means a "
    "non-adversarial prompt edit alone shifted measured behavior, independent of any model change.",
)
prompt_drift

<a id="reporting-template"></a>
## 📝 Testing Report

Built from this run's data through the same [uniform HTML template](../reporting/templates/scenario_report.html.j2) every scenario in this repo uses: **Executive Summary → Key Findings → Testing Scope → Testing Approach → Results Summary → High-Risk Cases (if any) → Next Steps → Appendix.** The Appendix carries the noise-floor table, every results table above, and a live-checked artifact trail.

In [ ]:
saved_paths = scenario.save_artifacts(
    sweep_results, noise_floor, sweep_drift, floating_results, floating_drift,
    control_results, unchanged_drift, corrupted_drift, prompt_results, prompt_drift,
    public_sweep_results, public_noise_floor, public_sweep_drift,
    public_floating_results, public_floating_drift,
)
artifacts_table = artifact_trail(scenario.artifacts(saved_paths))

charts = [c for c in [trajectory_chart, public_trajectory_chart, control_chart, prompt_chart] if c is not None]
report = scenario.build_report(
    sequence, floating_entry, noise_floor, sweep_drift, floating_drift,
    unchanged_drift, corrupted_drift, prompt_drift, charts, artifacts_table,
    public_sweep_drift, public_floating_drift,
)

html = render_report(report)
report_path = save_report(html, "outputs/reports/drift_detection.html")
print(f"Report saved to {report_path}")
embed_report(html)

<a id="how-to-extend"></a>
## 🔧 How to Extend This Scenario

- **Widen the lineage** — `DRIFT_MODEL_SEQUENCE` (`.env`) takes any number of `ISO_DATE:deployment` pairs ≥ 2; a longer-lived or explicitly version-pinned deployment tier would let this run a richer trajectory than the 2 points that survived retirement this time.
- **Run it on a cadence** — this notebook demonstrates the mechanism works once; the design doc's actual "per release / set cadence" repeat loop (`docs/drift_detection.md`) needs this wired to run automatically, not just on demand.
- **Test more than one prompt edit** — `scenario.PROMPT_DRIFT_V2_SYSTEM_PROMPT` is a single candidate rewrite; a small library of plausible edits (reordering, tone changes, added constraints) would show whether material drift from prompt changes is common, or this one rewrite happened to be unlucky.
- **Add a tool/API response-drift track** — a fourth drift axis discussed but not yet built: the agent's own reasoning stays fixed, but a tool it calls changes its response shape or values. Needs a tool-calling use case this golden set doesn't have; Consistency & Reliability's agentic harness (`adapters/agent_otel.py`) is the closest existing infrastructure to start from.
- **Stress-test the material-drift threshold further** — `scenario.INJECTED_DRIFT_SYSTEM_SUFFIX` is a deliberately blunt corruption (doubles every number); a subtler perturbation would test whether the two-sample z-test's threshold (even after BH correction) is sensitive enough to catch a more realistic drift event, not just an obvious one.
- **Add a dedicated NLI model for the entailment check** — same open item as Consistency & Reliability: `bidirectional_entailment` uses an LLM-judge prompt rather than a dedicated NLI model, which removes one source of judge-model variance from the semantic-drift signal at the cost of a new ML dependency.
- **Different target system** — everything above is wired to Intended Performance's golden set and RAG mandate; a different simulated system would need `run_version_sweep`'s `_run_once` closure (and `run_prompt_drift_check`'s) pointed at a different scenario module's run/score functions instead.